In [ ]:
import os
import torch
import logging
import numpy as np
import netCDF4 as nc
from functools import partial
from typing import Any, Callable, Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import xarray as xr
import dask
import cartopy.crs as ccrs

In [ ]:
import torch

# Check if CUDA (NVIDIA GPU) is available
cuda_available = torch.cuda.is_available()

if cuda_available:
    print("CUDA (GPU) is available.")
    print("Number of CUDA devices:", torch.cuda.device_count())
    print("CUDA Device Name:", torch.cuda.get_device_name(0)) # Get name of the first GPU
else:
    print("No CUDA (GPU) available. PyTorch will use CPU.")

# Load dataset

In [ ]:
import xarray as xr

# Load dataset from Zarr
train_dataset = xr.open_zarr("../data/train_dataset_monthly_radiation_LWP_no_poles.zarr", consolidated=True)
val_dataset = xr.open_zarr("../data/val_dataset_monthly_radiation_LWP_no_poles.zarr", consolidated=True)
test_dataset  = xr.open_zarr("../data/test_dataset_monthly_radiation_LWP_no_poles.zarr", consolidated=True)
#test_dataset  = xr.open_zarr("/glade/work/asampath/cesm_backup/AS_val_dataset_normalized_by_5member_monthly_adjacent_timestep_TGCLDLWP.zarr", consolidated=True)
static_ds = xr.open_zarr("../data/static_input_no_poles.zarr", consolidated=False)
static_input_tensor = torch.tensor(static_ds["static_input"].values, dtype=torch.float32).unsqueeze(0)
static_repeated = static_input_tensor.repeat(2, 1, 1, 1)

In [ ]:
from torch.utils.data import Dataset
class XarrayBalancedBatchDataset_monthly(Dataset):
    def __init__(self, xarray_ds, shuffle=True):
        """
        Dataset that samples 10 label=0 and 10 label=1 samples
        for each (month, member) combination.
        """
        self.data = xarray_ds
        self.labels = self.data["labels"].values
        self.months = self.data["month"][:,0].values
        self.members = self.data["member"].values

        # Create list of unique (month, member) pairs
        month_member_pairs = list(zip(self.months, self.members))
        self.unique_pairs = list(set(month_member_pairs))

        # Shuffle for randomness
        if shuffle:
            np.random.seed(42)
            np.random.shuffle(self.unique_pairs)

    def __len__(self):
        return len(self.unique_pairs)

    def __getitem__(self, idx):
        """
        Returns 20 indices from the same (month, member), balanced by label.
        """
        month_value, member_value = self.unique_pairs[idx]

        # Filter for the current (month, member)
        mask = (self.months == month_value) & (self.members == member_value)

        # Separate by labels
        zero_indices = np.flatnonzero((self.labels == 0) & mask)
        one_indices = np.flatnonzero((self.labels == 1) & mask)
        selected_indices = np.concatenate([zero_indices, one_indices])
        return selected_indices

In [ ]:
# Full version of XarrayBalancedBatchPrefetcher compatible with BatchForecastLenDataLoader

from torch.utils.data import Dataset
from multiprocessing import Process, Queue, Event, Manager
from threading import Thread
import numpy as np
import torch
import time
import signal

class XarrayBalancedBatchPrefetcher(Dataset):
    def __init__(self, xarray_ds, static_repeated, shuffle=True, num_workers=4, prefetch_factor=8):
        self.data = xarray_ds
        self.labels = self.data["labels"].values
        self.months = self.data["month"].values[:, 0]  # use month[:, 0] as base time
        self.members = self.data["member"].values
        self.static_repeated = static_repeated

        all_pairs = list(set(zip(self.months, self.members)))
        all_pairs.sort()
        self.unique_pairs = all_pairs
        print(len(self.unique_pairs))
        if shuffle:
            np.random.seed(42)
            np.random.shuffle(self.unique_pairs)

        self.num_workers = num_workers
        self.prefetch_factor = prefetch_factor
        self.prefetch_queue = Queue(maxsize=prefetch_factor)
        self.stop_event = Event()
        self.manager = Manager()
        self.results = self.manager.dict()

        signal.signal(signal.SIGTERM, self.handle_signal)

        self.prefetch_thread = Thread(target=self.prefetch_batches, daemon=True)
        self.prefetch_thread.start()

    def set_epoch(self, epoch):
        print(f"set_epoch({epoch}) called")

        self.stop_event.set()
        if hasattr(self, "prefetch_thread") and self.prefetch_thread is not None:
            if self.prefetch_thread.is_alive():
                self.prefetch_thread.join(timeout=5)

        self.stop_event = Event()
        self.prefetch_queue = Queue(maxsize=self.prefetch_factor)

        try:
            if hasattr(self, "manager") and self.manager is not None:
                self.manager.shutdown()
        except Exception as e:
            print(f"Warning: failed to shutdown manager: {e}")

        self.manager = Manager()
        self.results = self.manager.dict()

        self.prefetch_thread = Thread(target=self.prefetch_batches, daemon=True)
        self.prefetch_thread.start()

    def __len__(self):
        return len(self.unique_pairs)

    def __getitem__(self, _):
        #return self.prefetch_queue.get()
        try:
                return self.prefetch_queue.get(timeout=5)  # wait up to 5 sec
        except Exception as e:
            print("Prefetch queue timeout:", e)
            raise StopIteration  # signal end of data

    def handle_signal(self, signum, frame):
        print("Received termination signal. Stopping prefetching...")
        self.stop_event.set()

    def prefetch_batches(self):
        idx = 0
        try:
            while not self.stop_event.is_set():
                if not self.prefetch_queue.full():
                    chunk = self.unique_pairs[idx:idx + self.num_workers]
                    if self.results is None:
                        if self.manager is None:
                            self.manager = Manager()
                        self.results = self.manager.dict()
                    else:
                        self.results.clear()
                    processes = [
                        Process(target=self.worker_process, args=(i, pair, self.results))
                        for i, pair in enumerate(chunk)
                    ]
                    for p in processes:
                        p.start()
                    for p in processes:
                        p.join()

                    for i in range(len(chunk)):
                        if self.results.get(i) is not None:
                            self.prefetch_queue.put(self.results[i])
                    idx += self.num_workers
                else:
                    time.sleep(0.05)
        except Exception as e:
            print(f"Prefetching error: {e}")
    '''
    def worker_process(self, k, pair, result_dict):
        try:
            month_value, member_value = pair
            mask = (self.months == month_value) & (self.members == member_value)
            zero_indices = np.flatnonzero((self.labels == 0) & mask)
            one_indices = np.flatnonzero((self.labels == 1) & mask)

            selected_indices = np.concatenate([zero_indices, one_indices])
            if len(selected_indices) == 0:
                result_dict[k] = None
                return

            data_batch = self.data["data"].isel(pairs=selected_indices).load()
            x1 = torch.from_numpy(data_batch[:, 0].values).float()
            x2 = torch.from_numpy(data_batch[:, 1].values).float()

            x1_combined = torch.cat([x1, self.static_repeated], dim=1)
            x2_combined = torch.cat([x2, self.static_repeated], dim=1)

            result_dict[k] = {"x1": x1_combined, "x2": x2_combined}

        except Exception as e:
            print(f"Worker error for pair {pair}: {e}")
            result_dict[k] = None
    '''
    def worker_process(self, k, pair, result_dict):
        try:
            month_value, member_value = pair
            mask = (self.months == month_value) & (self.members == member_value)
            zero_indices = np.flatnonzero((self.labels == 0) & mask)
            one_indices = np.flatnonzero((self.labels == 1) & mask)

            selected_indices = np.concatenate([zero_indices, one_indices])
            if len(selected_indices) == 0:
                result_dict[k] = None
                return

            data_batch = self.data["data"].isel(pairs=selected_indices).load()
            label_batch = torch.tensor(self.labels[selected_indices], dtype=torch.long)

            x1 = torch.from_numpy(data_batch[:, 0].values).float()
            x2 = torch.from_numpy(data_batch[:, 1].values).float()

            x1_combined = torch.cat([x1, self.static_repeated], dim=1)
            x2_combined = torch.cat([x2, self.static_repeated], dim=1)
            
            member=self.data.member.isel(pairs=selected_indices).load()

            result_dict[k] = {
                "x1": x1_combined,
                "x2": x2_combined,
                "x1_labels": label_batch,
                "x2_labels": label_batch,
                "member":member.values.tolist(),
                "month":month_value
            }

        except Exception as e:
            print(f"Worker error for pair {pair}: {e}")
            result_dict[k] = None
        
    def __del__(self):
        try:
            print("Cleaning up...")
            self.stop_event.set()
            if hasattr(self, "prefetch_thread") and self.prefetch_thread.is_alive():
                self.prefetch_thread.join(timeout=5)
            if hasattr(self, "manager"):
                try:
                    self.manager.shutdown()
                except Exception as e:
                    print(f"Manager shutdown error: {e}")
            if hasattr(self, "prefetch_queue"):
                while not self.prefetch_queue.empty():
                    try:
                        self.prefetch_queue.get_nowait()
                    except Exception:
                        break
            print("Cleanup complete.")
        except Exception as e:
            print(f"Cleanup error: {e}")

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.__del__()

    @property
    def forecast_len(self):
        return 0

    def batches_per_epoch(self):
        return len(self)


class BatchForecastLenDataLoader:
    def __init__(self, dataset):
        self.dataset = dataset
        self.forecast_len = dataset.forecast_len + 1

    '''
    def __iter__(self):
        dataset_iter = iter(self.dataset)
        for _ in range(len(self)):
            yield next(dataset_iter)
    '''
    def __iter__(self):
        dataset_iter = iter(self.dataset)
        max_batches = len(self)  # total batches expected
        for _ in range(max_batches):
            try:
                yield next(dataset_iter)
            except StopIteration:
                break

    def __len__(self):
        if hasattr(self.dataset, "batches_per_epoch"):
            return self.dataset.batches_per_epoch() * self.forecast_len
        else:
            return len(self.dataset) * self.forecast_len


In [ ]:
import torch
import xarray as xr


# Load and prepare static input
static_ds = xr.open_zarr("../data/static_input_no_poles.zarr", consolidated=False)
static_input_tensor = torch.tensor(static_ds["static_input"].values, dtype=torch.float32).unsqueeze(0)
static_repeated = static_input_tensor.repeat(2, 1, 1, 1)

# Initialize datasets
train_dataset = XarrayBalancedBatchPrefetcher(
    xarray_ds=train_dataset,
    static_repeated=static_repeated,
    shuffle=True,
    num_workers=8,
    prefetch_factor=8,
)

val_dataset = XarrayBalancedBatchPrefetcher(
    xarray_ds=val_dataset,
    static_repeated=static_repeated,
    shuffle=False,
    num_workers=8,
    prefetch_factor=8,
)
test_dataset = XarrayBalancedBatchPrefetcher(
    xarray_ds=test_dataset,
    static_repeated=static_repeated,
    shuffle=False,
    num_workers=8,
    prefetch_factor=8,
)

# Wrap in batch loaders
#train_loader = BatchForecastLenDataLoader(train_dataset)
#val_loader = BatchForecastLenDataLoader(val_dataset)
test_loader = BatchForecastLenDataLoader(test_dataset)


# XAI Analaysis

In [ ]:
# correct one
import copy
from tqdm import tqdm
from captum.attr import IntegratedGradients
import torch.nn.functional as F
results = []  # Will hold dicts of PMM results with metadata

# 🔹 **Define Euclidean Distance Loss**
def euclidean_distance_loss(x1, x2):
    """Computes Euclidean distance loss between x1 and x2 embeddings."""
    z_i = model(x1)  # Feature embeddings for x1
    z_j = model(x2)  # Feature embeddings for positive pair x2

    # Compute Euclidean distance
    loss = F.pairwise_distance(z_i, z_j, p=2).mean()  # L2 distance
    return loss.unsqueeze(0)  # Keep batch dimension

model.to(device)
static_repeated = static_input_tensor.repeat(2, 1, 1, 1).to(device, non_blocking=True)
count = 0
for batch in tqdm(test_loader, desc="Integrated gradient"):
    PPE_member = batch["member"][:]
    month = batch["month"]
    #if month.startswith('0002') or month.startswith('0003'):
    #    continue
    x1 = batch["x1"].to(device, non_blocking=True)         # [N, C, H, W]
    labels1 = batch["x1_labels"].cpu().numpy()             # [N]
    '''
    x1_combined = torch.cat([x1, static_repeated], dim=1)
    x2_combined = torch.cat([x2, static_repeated], dim=1)
    '''
    x1_combined = torch.cat([x1, static_input_tensor], dim=1)
    # Integrated Gradients
    ig = IntegratedGradients(euclidean_distance_loss) 
    attributions, delta = ig.attribute(
        (x1[:1], x1[1:]),
        baselines=(x1[1:], x1[1:]), 
        return_convergence_delta=True
    )

    # Split into ML and KK (custom logic—assumes index=1 for now)
    index = 1
    # Gradients
    gradient_ML = attributions[0].detach().cpu().numpy()
    gradient_KK = attributions[1].detach().cpu().numpy()
    mean_gradient_ML = gradient_ML.mean(axis=0)  # Mean over batch
    mean_gradient_KK = gradient_KK.mean(axis=0)

    # Inputs
    inputs_ML = x1[:index].detach().cpu().numpy()
    inputs_KK = x1[index:].detach().cpu().numpy()
    mean_inputs_ML = inputs_ML.mean(axis=0)
    mean_inputs_KK = inputs_KK.mean(axis=0)


    # Store results
    results.append({
        "month": month.item() if hasattr(month, "item") else month,
        "PPE_member": PPE_member[0].item() if hasattr(PPE_member, "item") else PPE_member[0],
        "mean_gradient_ML": mean_gradient_ML,
        "mean_gradient_KK": mean_gradient_KK,
        "mean_inputs_ML": mean_inputs_ML,
        "mean_inputs_KK": mean_inputs_KK,
    })
    count += 1
    if count%100 == 0:
        print(count)
        

# Sort results by week and PPE_member
sorted_results = sorted(results, key=lambda x: (x["month"], x["PPE_member"]))


In [ ]:
import pickle

with open("sorted_results_VGG_monthly_radiation_LWP_no_poles.pkl", "wb") as f:
    pickle.dump(sorted_results, f)

In [ ]:
import pickle
with open("sorted_results_VGG_monthly_radiation_LWP_no_poles.pkl", "rb") as f:
    sorted_results = pickle.load(f)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

plt.rcParams['font.size'] = 11

PLOT_VARS = ['LWCF', 'SWCF', 'FLUT', 'FSNT', 'TGCLDLWP']

def _extract(arr_or_tuple):
    if isinstance(arr_or_tuple, (list, tuple)) and hasattr(arr_or_tuple[0], "shape"):
        return np.asarray(arr_or_tuple[0])
    return np.asarray(arr_or_tuple)

def plot_mean_grads_tau_ml_row(results, var_names_key='var_names', plot_vars=PLOT_VARS):
    """
    Plot 5-variable mean gradients (TAU-ML) sorted by max |gradient|.
    Each panel uses the same global color scale.
    """
    # ---- gather & mean across cases ----
    g_ML = np.stack([_extract(r['mean_gradient_ML']) for r in results]).mean(axis=0)  # (nvar, lat, lon)
    var_names = results[0].get(var_names_key, plot_vars)

    # ---- pick variables ----
    idx = [var_names.index(v) for v in plot_vars if v in var_names]
    row_names = [var_names[i] for i in idx]

    # ---- compute max |gradient| for each variable ----
    max_abs_grad = [np.nanmax(np.abs(g_ML[i])) for i in idx]

    # ---- reorder by descending |grad| magnitude ----
    order = np.argsort(max_abs_grad)[::-1]  # largest first
    idx = [idx[i] for i in order]
    row_names = [row_names[i] for i in order]
    max_abs_grad = [max_abs_grad[i] for i in order]
    print("Variable order by max |gradient|:", list(zip(row_names, max_abs_grad)))

    # ---- global color scale ----
    lim = float(np.nanpercentile(np.abs(g_ML[idx]), 99.5))
    norm = Normalize(vmin=-lim, vmax=+lim)

    # ---- layout ----
    nrows, ncols = 1, len(idx)
    fig = plt.figure(figsize=(20, 4.5), dpi=300)
    gs = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.15, wspace=0.13)

    for k, vi in enumerate(idx):
        ax = fig.add_subplot(gs[0, k], projection=ccrs.PlateCarree())

        im = ax.pcolormesh(
            lon_grid[26:-26], lat_grid[26:-26], g_ML[vi],
            cmap='coolwarm', norm=norm,
            transform=ccrs.PlateCarree(), shading='auto'
        )
        #ax.set_title(f"{row_names[k]}", fontsize=10, fontweight='bold')  # <-- puts title at top
        ax.set_aspect(1.3)

        # Panel title with variable name + max |grad|
        #label = f"[{row_names[k]}]\nmax|∇|={max_abs_grad[k]:.1e}"
        label = f"{row_names[k]}"
        ax.text(0.02, 0.98, label,
                transform=ax.transAxes, ha='left', va='top',
                fontsize=10, weight='bold',
                bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2'))

        # Map decoration
        ax.coastlines(linewidth=0.6)
        gl = ax.gridlines(draw_labels=True, x_inline=False, y_inline=False,
                          linewidth=0.4, color='gray', alpha=0.5)
        gl.xformatter = LONGITUDE_FORMATTER
        gl.yformatter = LATITUDE_FORMATTER
        gl.xlabel_style = {'size': 8}
        gl.ylabel_style = {'size': 8}
        try:
            gl.left_labels   = (k == 0)
            gl.right_labels  = False
            gl.top_labels    = False
            gl.bottom_labels = True
        except AttributeError:
            gl.ylabels_left   = (k == 0)
            gl.ylabels_right  = False
            gl.xlabels_top    = False
            gl.xlabels_bottom = True

    # ---- one shared colorbar ----
    cbar_ax = fig.add_axes([0.92, 0.353, 0.015, 0.30])
    cb = plt.colorbar(im, cax=cbar_ax)
    cb.ax.tick_params(labelsize=9)
    cb.set_label("Gradient", fontsize=11)
    
    # Force scientific notation to appear as 10^(-x)
    cb.formatter.set_powerlimits((0, 0))  # always show scientific
    cb.ax.yaxis.get_offset_text().set_visible(False)  # hide default 1e-5 text

    # Custom scientific notation in LaTeX style
    exp_str = int(np.floor(np.log10(lim)))
    cb.ax.text(0.85, 1, f"$10^{{{exp_str}}}$", transform=cb.ax.transAxes,
               ha='center', va='bottom', fontsize=11)
    cb.update_ticks()

    plt.tight_layout(rect=[0.03, 0.1, 0.90, 0.93])
    fig.suptitle("Mean Gradient (TAU-ML) — Sorted by max |gradient|",
                 fontsize=15, y=0.98)
    plt.show()

# usage
plot_mean_grads_tau_ml_row(sorted_results)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

plt.rcParams['font.size'] = 11

def _extract(arr_or_tuple):
    if isinstance(arr_or_tuple, (list, tuple)) and hasattr(arr_or_tuple[0], "shape"):
        return np.asarray(arr_or_tuple[0])
    return np.asarray(arr_or_tuple)

def plot_mean_grads_tau_ml_row(results):
    g_ML_full = np.stack([_extract(r['mean_gradient_ML']) for r in results]).mean(axis=0)  # (C,H,W)

    PLOT_VARS_NEW = ['SWCF', 'TGCLDLWP', 'FSNT', 'FLUT', 'LWCF']
    OLD_ORDER     = ['LWCF', 'SWCF', 'FLUT', 'FSNT', 'TGCLDLWP']

    # assume first 5 channels follow OLD_ORDER; reorder to PLOT_VARS_NEW
    g_ML_dyn = g_ML_full[:5]
    reorder_idx = [OLD_ORDER.index(v) for v in PLOT_VARS_NEW]
    g_ML = np.stack([g_ML_dyn[i] for i in reorder_idx], axis=0)
    row_names = PLOT_VARS_NEW

    lim = float(np.nanpercentile(np.abs(g_ML), 99.5))
    norm = Normalize(vmin=-lim, vmax=+lim)

    nrows, ncols = 1, len(row_names)
    fig = plt.figure(figsize=(20, 4.5), dpi=300)
    gs = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.15, wspace=0.13)

    for k in range(ncols):
        ax = fig.add_subplot(gs[0, k], projection=ccrs.PlateCarree())
        im = ax.pcolormesh(lon_grid[26:-26], lat_grid[26:-26], g_ML[k],
                           cmap='coolwarm', norm=norm,
                           transform=ccrs.PlateCarree(), shading='auto')
        ax.set_aspect(1.3)
        #ax.set_title(row_names[k], fontsize=10, fontweight='bold')
        ax.coastlines(linewidth=0.6)
        label = f"{row_names[k]}"
        ax.text(0.02, 0.98, label,
                transform=ax.transAxes, ha='left', va='top',
                fontsize=10, weight='bold',
                bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2'))
        gl = ax.gridlines(draw_labels=True, x_inline=False, y_inline=False,
                          linewidth=0.4, color='gray', alpha=0.5)
        gl.xformatter = LONGITUDE_FORMATTER; gl.yformatter = LATITUDE_FORMATTER
        gl.xlabel_style = {'size': 8}; gl.ylabel_style = {'size': 8}
        try:
            gl.left_labels   = (k == 0); gl.right_labels = False
            gl.top_labels    = False;     gl.bottom_labels = True
        except AttributeError:
            gl.ylabels_left   = (k == 0); gl.ylabels_right  = False
            gl.xlabels_top    = False;    gl.xlabels_bottom = True

    cbar_ax = fig.add_axes([0.92, 0.353, 0.015, 0.30])
    cb = plt.colorbar(im, cax=cbar_ax)
    cb.ax.tick_params(labelsize=9)
    cb.set_label("Gradient", fontsize=11)
    cb.formatter.set_powerlimits((0, 0))
    cb.ax.yaxis.get_offset_text().set_visible(False)
    exp_str = int(np.floor(np.log10(lim))) if lim > 0 else 0
    cb.ax.text(0.85, 1, f"$10^{{{exp_str}}}$", transform=cb.ax.transAxes,
               ha='center', va='bottom', fontsize=11)
    cb.update_ticks()

    plt.tight_layout(rect=[0.03, 0.1, 0.90, 0.93])
    fig.suptitle("Mean Gradient (TAU-ML vs normalized OBS) — New Order",
                 fontsize=15, y=0.98)
    plt.show()

# ✅ usage
plot_mean_grads_tau_ml_row(sorted_results)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

plt.rcParams['font.size'] = 11

def _extract(arr_or_tuple):
    if isinstance(arr_or_tuple, (list, tuple)) and hasattr(arr_or_tuple[0], "shape"):
        return np.asarray(arr_or_tuple[0])
    return np.asarray(arr_or_tuple)

def plot_zonal_mean_grads_tau_ml(results, lat_grid):
    # --- Stack and reorder gradients ---
    g_ML_full = np.stack([_extract(r['mean_gradient_ML']) for r in results]).mean(axis=0)  # (C,H,W)

    PLOT_VARS_NEW = ['SWCF', 'TGCLDLWP', 'FSNT', 'FLUT', 'LWCF']
    OLD_ORDER     = ['LWCF', 'SWCF', 'FLUT', 'FSNT', 'TGCLDLWP']

    # reorder to match PLOT_VARS_NEW
    g_ML_dyn = g_ML_full[:5]
    reorder_idx = [OLD_ORDER.index(v) for v in PLOT_VARS_NEW]
    g_ML = np.stack([g_ML_dyn[i] for i in reorder_idx], axis=0)  # (5, H, W)

    # --- Compute zonal means (mean over longitude) ---
    g_zonal = np.nanmean(g_ML, axis=-1)  # (5, H)
    lim = float(np.nanpercentile(np.abs(g_zonal), 99.5))
    norm = Normalize(vmin=-lim, vmax=+lim)

    # --- Plot zonal means ---
    fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
    colors = plt.cm.tab10(np.linspace(0, 1, len(PLOT_VARS_NEW)))
    for i, var in enumerate(PLOT_VARS_NEW):
        ax.plot(lat_grid[26:-26], g_zonal[i, :], label=var, color=colors[i], lw=2)

    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel("Latitude")
    ax.set_ylabel("Zonal-Mean Gradient")
    ax.set_title("Zonal Mean of Mean Gradients (TAU-ML vs Normalized OBS)")
    ax.grid(True, linestyle='--', alpha=0.5)

    # Deduplicate legend entries and place outside
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(),
              fontsize=10, frameon=False, loc='center left',
              bbox_to_anchor=(1.02, 0.5))
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()

# ✅ usage
plot_zonal_mean_grads_tau_ml(sorted_results, lat_grid)
